<a href="https://colab.research.google.com/github/svetlanama/ai_practice/blob/dev/DZ_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Завдання

* Створіть Reader
* Створіть датасет та розділіть його на тренувальні та тестові дані
* Виберіть метрики для поріняння якості моделей
* На основі метрик виберіть найкращу модель



In [1]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/HalyshAnton/IT-Step-Pyton-AI/main/module7/data/ratings.csv")

df.head()

,user_id,movie_id,rating,timestamp
0,172,94969,5.0,1396067836
1,172,98956,4.0,1396067879
2,176,73881,4.0,1499807147
3,221,1900,4.5,1288550866
4,333,33688,4.0,1412015122


In [2]:
!pip install -q surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from surprise import Dataset, SVD, Reader

reader = Reader(rating_scale=(0.5, 5))

data = Dataset.load_from_df(df[["user_id", "movie_id", "rating"]], reader)
data

In [5]:

from surprise import Dataset, SVD, Reader

reader = Reader(rating_scale=(0.2, 5))

data = Dataset.load_from_df(df[["user_id", "movie_id", "rating"]], reader)
data


In [6]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, train_size=0.7)

In [7]:
from surprise import BaselineOnly

bsl_options = {'method': 'als',
               'reg_u': 0.001,
               'reg_i': 0.001}

algo = BaselineOnly(bsl_options=bsl_options)
algo.fit(trainset)

Estimating biases using als...


In [8]:
from surprise import SVD

alfo = SVD(n_factors=80,
           n_epochs=40,
           )

algo.fit(trainset)

Estimating biases using als...


In [9]:
from surprise import KNNBasic

algo = KNNBasic(k=10,
                min_k=1,
                sim_options={'name': 'cosine',
                             'user_based': True})

algo.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [10]:
from surprise import CoClustering

algo = CoClustering(n_cltr_u = 5,
                    n_cltr_i = 15,
                    n_epochs = 50)

algo.fit(trainset)

In [11]:
from surprise import BaselineOnly, accuracy


bsl_options = {'method': 'als',
               'reg_u': 0.001,
               'reg_i': 0.001}

algo = BaselineOnly(bsl_options=bsl_options)
algo.fit(trainset)

preds = algo.test(testset)

print(f"mae = {accuracy.mae(preds, verbose=False)}")
print(f"mse = {accuracy.mse(preds, verbose=False)}")
print(f"rmse= {accuracy.rmse(preds, verbose=False)}")
print(f"fcp = {accuracy.fcp(preds, verbose=False)}")

Estimating biases using als...
mae = 0.7020401454854119
mse = 0.9116803364971001
rmse= 0.9548195308523492
fcp = 0.6261201403104439


In [12]:
algo.predict(uid=2,
             iid=98)

Prediction(uid=2, iid=98, r_ui=None, est=3.4600308391494887, details={'was_impossible': False})

In [13]:
# Model init
models = {
    "SVD": SVD(),
    "BaselineOnly": BaselineOnly(),
    "KNNBasic": KNNBasic()
}

# test
results = {}
for model_name, model in models.items():
    model.fit(trainset)
    predictions = model.test(testset)

    # quality estimation
    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)

    results[model_name] = {"RMSE": rmse, "MAE": mae}

# results
for model_name, metrics in results.items():
    print(f"{model_name}: RMSE = {metrics['RMSE']:.4f}, MAE = {metrics['MAE']:.4f}")

Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
SVD: RMSE = 0.9859, MAE = 0.7476
BaselineOnly: RMSE = 1.0025, MAE = 0.7590
KNNBasic: RMSE = 1.0888, MAE = 0.8305
